In [220]:
import bs4
import time
from datetime import date
import calendar
import pandas as pd
from selenium import webdriver
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC

## What my goal is from doing this Project?
What data do i want to get???<br>
I will scrape the data from Autotrader with Selenium and BeauttifulSoup.<br>
My goal is to get Make Year, Make, Model, Odometer, gearBox, Type of Fuel.<br>
I want 500 cars data

### I am just Trying to find the item that i have to find:

All of the item that i want our store in tag named **article**.<br>
I know how to get the data I store them I know where i can find the data i want to find and now i have to insert them in a list so at the end i can make a dataframe with them.

In [221]:
## I need to write a function which get a webdrive and return all the data nice and clean
def cleanAndInsert(article):
    carList = []
    for i in article:
        html_file = i.get_attribute("outerHTML")
        soup = bs4.BeautifulSoup(html_file,"html.parser")
        new_car=dict()
        article = soup.find("article")
        year=article.find("h2").find_all("span")[0].text.split()[0]## Year
        make=article.find("h2").find_all("span")[0].text.split()[1] ##Make
        model=article.get("data-model") #Data - model
        miliage=article.get("data-mileage") ## Miliage
        price=article.get("data-price")
        information=article.find("h2").find_all("span")[1].text ##More Information 
        new_car={"Year":year,"Make":make,"Model":model,"Miliage":miliage,"Price":price,"Information":information}
        carList.append(new_car)
    return carList


In [ ]:
df = pd.DataFrame()
brower = webdriver.Safari()
brower.get("https://www.autotrader.ca/lst?atype=C&cy=CA&desc=0&lat=43.81800842285156&lon=-79.42473602294922&offer=N%2CU&page=1&search_id=xcqjyqqjv3&sort=standard&source=listpage_pagination&ustate=N%2CU&zip=L3T5L8%20Thornhill%2C%20ON&zipr=100")
article = brower.find_elements(By.CSS_SELECTOR,"article") #All of the datas that come back are a webdrive objects
df=pd.concat([df, pd.DataFrame(cleanAndInsert(article))],ignore_index=True)
next_buttom = brower.find_element(By.CSS_SELECTOR,"button[aria-label='Go to next page']")
next_buttom.click()
time.sleep(10)
article = brower.find_elements(By.CSS_SELECTOR,"article")
df=pd.concat([df, pd.DataFrame(cleanAndInsert(article))],ignore_index=True)

### What is the next Part:
I have to find the Next page buutom and try to automate it in a *loop* to go through at least 20 pages.<br>
There is a next buttom that i can use to go to next page and load their data.<br>

In [225]:
brower = webdriver.Safari()
brower.get("https://www.autotrader.ca/lst?atype=C&cy=CA&desc=0&lat=43.81800842285156&lon=-79.42473602294922&offer=N%2CU&page=1&search_id=xcqjyqqjv3&sort=standard&source=listpage_pagination&ustate=N%2CU&zip=L3T4S3%20Thornhill%2C%20ON&zipr=100")
df=pd.DataFrame()
for i in range(19):
    time.sleep(10)
    article = brower.find_elements(By.CSS_SELECTOR,"article") #All of the datas that come back are a webdrive objects
    df=pd.concat([df, pd.DataFrame(cleanAndInsert(article))],ignore_index=True)
    next_buttom = brower.find_element(By.CSS_SELECTOR,"button[aria-label='Go to next page']")
    next_buttom.click()

today=date.today()
df.to_csv(f'Data/{today.day}-{calendar.month_name[today.month]}-{today.year}.csv')
